# ARES-Hybrid-INN — Full Colab Training (CIFAR-10 + SIPI + ImageNet-256)

**What this notebook does**
1. Mount Google Drive and save **every** checkpoint / log there
2. Download **CIFAR-10**, **SIPI** misc volume, try **ImageNet-256 (Kaggle)**
3. Build train / val / test splits (hash-safe)
4. **Resume** from existing `ares_hybrid_inn_best.pt` if present (no training from scratch unless you choose)
5. Train ARES-Hybrid-INN (CNN + INN + residual) with AMP on GPU
6. Keep **best** model by validation loss/PSNR proxy on Drive

**Runtime:** Colab GPU (T4/A100 preferred). High-RAM runtime recommended.

**Drive layout**
```
MyDrive/ARES_Hybrid_INN/
  checkpoints/   ares_hybrid_inn_best.pt, latest, epoch_*.pt
  datasets/      cifar10, sipi, imagenet256, manifest
  logs/          train_history.json
  exports/       best model copy for local use
```


In [ ]:
# ===== CELL 01: Mount Drive & paths =====
from google.colab import drive
drive.mount('/content/drive')

import os, json, sys, time, random, math, shutil, hashlib
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ARES_Hybrid_INN')
PATHS = {
    'root': DRIVE_ROOT,
    'checkpoints': DRIVE_ROOT / 'checkpoints',
    'datasets': DRIVE_ROOT / 'datasets',
    'logs': DRIVE_ROOT / 'logs',
    'exports': DRIVE_ROOT / 'exports',
    'code': Path('/content/ARES-Upgraded'),
}
for p in PATHS.values():
    Path(p).mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Checkpoints:', PATHS['checkpoints'])


In [ ]:
# ===== CELL 02: Install deps =====
!pip install -q torch torchvision pillow numpy tqdm matplotlib scikit-image cryptography kaggle 2>/dev/null
import torch
print('Torch', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# ===== CELL 03: Project source =====
# Prefer: zip already on Drive, or upload ARES-Hybrid-INN_Upgraded.zip
import zipfile
from pathlib import Path

CANDIDATES = [
    Path('/content/drive/MyDrive/ARES_Hybrid_INN/ARES-Hybrid-INN_Upgraded.zip'),
    Path('/content/drive/MyDrive/ARES_Upgraded/ARES-Hybrid-INN_Upgraded.zip'),
    Path('/content/drive/MyDrive/ARES_Upgraded/ARES-Upgraded_FINAL.zip'),
    Path('/content/ARES-Hybrid-INN_Upgraded.zip'),
]
src = next((p for p in CANDIDATES if p.exists()), None)
CODE = Path('/content/ARES-Upgraded')
if src:
    print('Extracting', src)
    with zipfile.ZipFile(src) as z:
        z.extractall('/content')
    # zip may contain ARES-Upgraded/ top folder
    if not CODE.exists():
        found = list(Path('/content').glob('**/models/hybrid/ares_hybrid_inn.py'))
        if found:
            CODE = found[0].parents[2]
    print('Code root:', CODE)
else:
    print('WARNING: Project zip not found on Drive.')
    print('Upload ARES-Hybrid-INN_Upgraded.zip to MyDrive/ARES_Hybrid_INN/ then re-run this cell.')
    print('Notebook will still embed a minimal Hybrid model for training.')

import sys
if CODE.exists():
    sys.path.insert(0, str(CODE))
print('sys.path[0]:', sys.path[0])


In [ ]:
# ===== CELL 04: Model definition (from project or fallback) =====
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image

try:
    from models.hybrid.ares_hybrid_inn import ARESHybridINN, load_hybrid
    from models.upgraded.ares_upgraded import compute_attention_map
    print('Loaded ARESHybridINN from project source')
    HAS_PROJECT = True
except Exception as e:
    print('Project import failed, using embedded Hybrid definition:', e)
    HAS_PROJECT = False

    def compute_attention_map(gray, block=8):
        h, w = gray.shape
        var_map = np.zeros((h, w), np.float32)
        step = max(1, block // 2)
        for i in range(0, h - block + 1, step):
            for j in range(0, w - block + 1, step):
                patch = gray[i:i+block, j:j+block].astype(np.float32)
                v = float(patch.var())
                var_map[i:i+block, j:j+block] = np.maximum(var_map[i:i+block, j:j+block], v)
        gx = np.abs(np.diff(gray.astype(np.float32), axis=1, prepend=gray[:, :1]))
        gy = np.abs(np.diff(gray.astype(np.float32), axis=0, prepend=gray[:1, :]))
        edge = gx + gy
        score = 0.65 * (var_map / (var_map.max() + 1e-6)) + 0.35 * (edge / (edge.max() + 1e-6))
        return np.clip(score, 0, 1).astype(np.float32)

    class ResidualBlock(nn.Module):
        def __init__(self, ch):
            super().__init__()
            self.c1 = nn.Conv2d(ch, ch, 3, padding=1)
            self.b1 = nn.BatchNorm2d(ch)
            self.c2 = nn.Conv2d(ch, ch, 3, padding=1)
            self.b2 = nn.BatchNorm2d(ch)
        def forward(self, x):
            r = F.relu(self.b1(self.c1(x)))
            return F.relu(x + self.b2(self.c2(r)))

    def _clamp_log_scale(s, clamp=2.0):
        return clamp * torch.tanh(s / clamp)

    class AffineCoupling(nn.Module):
        def __init__(self, channels, hidden=64, clamp=2.0):
            super().__init__()
            assert channels % 2 == 0
            self.half = channels // 2
            self.clamp = clamp
            self.net = nn.Sequential(
                nn.Conv2d(self.half, hidden, 3, padding=1), nn.ReLU(True),
                nn.Conv2d(hidden, hidden, 3, padding=1), nn.ReLU(True),
                nn.Conv2d(hidden, self.half * 2, 3, padding=1),
            )
            nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
        def forward(self, x, reverse=False):
            x1, x2 = x[:, :self.half], x[:, self.half:]
            s, t = self.net(x1).chunk(2, dim=1)
            s = _clamp_log_scale(s, self.clamp)
            if not reverse:
                y2 = x2 * torch.exp(s) + t
                return torch.cat([x1, y2], 1)
            y2 = (x2 - t) * torch.exp(-s)
            return torch.cat([x1, y2], 1)

    class InvertibleNetwork(nn.Module):
        def __init__(self, channels=8, n_blocks=4, hidden=64):
            super().__init__()
            self.blocks = nn.ModuleList([AffineCoupling(channels, hidden) for _ in range(n_blocks)])
        def forward(self, x, reverse=False):
            if not reverse:
                for b in self.blocks: x = b(x, False)
            else:
                for b in reversed(self.blocks): x = b(x, True)
            return x

    class ARESHybridINN(nn.Module):
        def __init__(self, base=32, inn_channels=8, n_inn_blocks=4):
            super().__init__()
            self.enc1 = nn.Sequential(nn.Conv2d(4, base, 3, padding=1), nn.ReLU(True), ResidualBlock(base))
            self.enc2 = nn.Sequential(nn.Conv2d(base, base*2, 3, stride=2, padding=1), nn.ReLU(True), ResidualBlock(base*2))
            self.bot = ResidualBlock(base*2)
            self.cond = nn.Sequential(nn.Linear(1, 16), nn.ReLU(True), nn.Linear(16, base), nn.ReLU(True))
            self.to_inn = nn.Conv2d(base*2 + base, inn_channels, 1)
            self.inn = InvertibleNetwork(inn_channels, n_inn_blocks, max(base, 32))
            self.from_inn = nn.Conv2d(inn_channels, base*2, 1)
            self.res_head = nn.Sequential(
                nn.ConvTranspose2d(base*2, base, 4, stride=2, padding=1), nn.ReLU(True),
                ResidualBlock(base), nn.Conv2d(base, 1, 3, padding=1), nn.Tanh())
            self.mask_head = nn.Sequential(
                nn.ConvTranspose2d(base*2, base, 4, stride=2, padding=1), nn.ReLU(True),
                nn.Conv2d(base, 1, 3, padding=1), nn.Sigmoid())
        def forward(self, cover_att, rate):
            e1 = self.enc1(cover_att)
            e2 = self.enc2(e1)
            b = self.bot(e2)
            c = self.cond(rate).unsqueeze(-1).unsqueeze(-1).expand(-1, -1, b.shape[2], b.shape[3])
            z = self.to_inn(torch.cat([b, c], 1))
            z = self.inn(z)
            feat = self.from_inn(z)
            return self.res_head(feat), self.mask_head(feat)

print('Model class ready')


In [ ]:
# ===== CELL 05: Dataset download helpers =====
import urllib.request, tarfile, zipfile, io

DATASET_MANIFEST = {}

def download_cifar10(root=None):
    root = Path(root or PATHS['datasets'] / 'cifar10')
    root.mkdir(parents=True, exist_ok=True)
    try:
        from torchvision.datasets import CIFAR10
        ds = CIFAR10(root=str(root), train=True, download=True)
        ds_t = CIFAR10(root=str(root), train=False, download=True)
        # export PNGs for unified loader
        out_train = root / 'images_train'
        out_test = root / 'images_test'
        out_train.mkdir(exist_ok=True)
        out_test.mkdir(exist_ok=True)
        def export(ds, out, limit=None):
            n = len(ds) if limit is None else min(limit, len(ds))
            for i in range(n):
                img, _ = ds[i]
                p = out / f'{i:05d}.png'
                if not p.exists():
                    img.save(p)
            return n
        # full set can be large on disk; export all by default
        ntr = export(ds, out_train)
        nte = export(ds_t, out_test)
        return {'status': 'OK', 'train_images': ntr, 'test_images': nte, 'path': str(root)}
    except Exception as e:
        return {'status': 'FAILED', 'error': str(e)}


def download_sipi(root=None):
    """SIPI Misc volume (classic test images)."""
    root = Path(root or PATHS['datasets'] / 'sipi')
    root.mkdir(parents=True, exist_ok=True)
    url = 'https://sipi.usc.edu/database/misc.zip'
    zip_path = root / 'misc.zip'
    try:
        if not zip_path.exists() or zip_path.stat().st_size < 1000:
            print('Downloading SIPI misc.zip ...')
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(root)
        files = list(root.rglob('*.tiff')) + list(root.rglob('*.tif')) + list(root.rglob('*.png'))
        # convert tiff -> png RGB
        out = root / 'images'
        out.mkdir(exist_ok=True)
        n = 0
        for f in files:
            try:
                im = Image.open(f).convert('RGB')
                im.save(out / f'{f.stem}.png')
                n += 1
            except Exception:
                pass
        return {'status': 'OK', 'num_files': n, 'path': str(out)}
    except Exception as e:
        return {'status': 'FAILED', 'error': str(e), 'num_files': 0}


def try_kaggle_imagenet256(root=None):
    """Requires Kaggle API credentials in Colab (kaggle.json)."""
    root = Path(root or PATHS['datasets'] / 'imagenet256')
    root.mkdir(parents=True, exist_ok=True)
    kaggle_json = Path('/root/.kaggle/kaggle.json')
    drive_kaggle = Path('/content/drive/MyDrive/kaggle.json')
    if not kaggle_json.exists():
        if drive_kaggle.exists():
            Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
            shutil.copy(drive_kaggle, kaggle_json)
            os.chmod(kaggle_json, 0o600)
        else:
            return {
                'status': 'SKIPPED',
                'reason': 'Place kaggle.json on Drive at MyDrive/kaggle.json or run: '
                          'from google.colab import files; files.upload()  # kaggle.json',
                'path': str(root),
            }
    try:
        # Popular community 256px ImageNet-style sets vary; try common dataset slug
        # User can change DATASET_SLUG below if needed
        DATASET_SLUG = os.environ.get('KAGGLE_IMAGENET_SLUG', 'ifigotin/imagenetmini-1000')
        print('Kaggle download:', DATASET_SLUG)
        import subprocess
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', str(root), '--unzip'],
            capture_output=True, text=True)
        if r.returncode != 0:
            return {'status': 'FAILED', 'error': r.stderr[-500:], 'path': str(root)}
        imgs = list(root.rglob('*.JPEG')) + list(root.rglob('*.jpg')) + list(root.rglob('*.png'))
        return {'status': 'OK', 'num_files': len(imgs), 'path': str(root), 'slug': DATASET_SLUG}
    except Exception as e:
        return {'status': 'FAILED', 'error': str(e), 'path': str(root)}

print('Download helpers ready')


In [ ]:
# ===== CELL 06: Download datasets =====
print('[CELL 06] Downloading CIFAR-10...')
DATASET_MANIFEST['cifar10'] = download_cifar10()
print('[CELL 06] CIFAR-10:', DATASET_MANIFEST['cifar10'].get('status'),
      'train', DATASET_MANIFEST['cifar10'].get('train_images'),
      'test', DATASET_MANIFEST['cifar10'].get('test_images'))

print('[CELL 06] Downloading SIPI...')
DATASET_MANIFEST['sipi'] = download_sipi()
print('[CELL 06] SIPI:', DATASET_MANIFEST['sipi'].get('status'),
      'files:', DATASET_MANIFEST['sipi'].get('num_files'))

print('[CELL 06] Attempting ImageNet-256 (Kaggle)...')
DATASET_MANIFEST['imagenet256'] = try_kaggle_imagenet256()
print('[CELL 06] ImageNet-256:', DATASET_MANIFEST['imagenet256'].get('status'))

manifest_path = PATHS['datasets'] / 'dataset_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(DATASET_MANIFEST, f, indent=2)
print('[CELL 06] Manifest written:', manifest_path)
print(json.dumps(DATASET_MANIFEST, indent=2)[:1500])


In [ ]:
# ===== CELL 07: Build image path list + splits =====
from torch.utils.data import Dataset, DataLoader

def collect_images():
    paths = []
    # CIFAR train
    p = Path(DATASET_MANIFEST.get('cifar10', {}).get('path', PATHS['datasets']/'cifar10'))
    paths += list((p/'images_train').glob('*.png')) if (p/'images_train').exists() else []
    # SIPI
    sipi = DATASET_MANIFEST.get('sipi', {})
    if sipi.get('status') == 'OK':
        paths += list(Path(sipi['path']).glob('*.png'))
    # ImageNet
    imn = DATASET_MANIFEST.get('imagenet256', {})
    if imn.get('status') == 'OK':
        root = Path(imn['path'])
        paths += list(root.rglob('*.JPEG'))[:20000]  # cap for Colab disk
        paths += list(root.rglob('*.jpg'))[:5000]
        paths += list(root.rglob('*.png'))[:5000]
    paths = [Path(x) for x in paths if Path(x).is_file()]
    # dedupe
    paths = sorted(set(paths), key=lambda x: str(x))
    return paths

all_paths = collect_images()
print('Total images collected:', len(all_paths))
if len(all_paths) < 100:
    print('WARNING: few images — CIFAR export may still be running or SIPI failed.')
    print('Training will still run on whatever is available + CIFAR tensors fallback.')

rng = random.Random(42)
rng.shuffle(all_paths)
n = len(all_paths)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
train_paths = all_paths[:n_train]
val_paths = all_paths[n_train:n_train+n_val]
test_paths = all_paths[n_train+n_val:]
print(f'Split train={len(train_paths)} val={len(val_paths)} test={len(test_paths)}')

split_info = {
    'n_total': n, 'n_train': len(train_paths), 'n_val': len(val_paths), 'n_test': len(test_paths),
    'seed': 42,
}
with open(PATHS['logs']/'split_info.json', 'w') as f:
    json.dump(split_info, f, indent=2)


In [ ]:
# ===== CELL 08: Dataset class =====
IMG_SIZE = 128  # use 128 for speed/VRAM; set 256 if you have A100 high RAM

class HybridImageDataset(Dataset):
    def __init__(self, paths, size=128, cifar_fallback=True):
        self.paths = list(paths)
        self.size = size
        self.cifar = None
        if len(self.paths) < 50 and cifar_fallback:
            try:
                from torchvision.datasets import CIFAR10
                self.cifar = CIFAR10(root=str(PATHS['datasets']/'cifar10'), train=True, download=True)
                print('Using CIFAR-10 tensor fallback, n=', len(self.cifar))
            except Exception as e:
                print('CIFAR fallback failed', e)

    def __len__(self):
        if self.paths:
            return len(self.paths)
        if self.cifar is not None:
            return len(self.cifar)
        return 1000

    def _load(self, idx):
        if self.paths:
            p = self.paths[idx % len(self.paths)]
            img = Image.open(p).convert('RGB')
        elif self.cifar is not None:
            img, _ = self.cifar[idx % len(self.cifar)]
        else:
            arr = np.random.randint(0, 256, (self.size, self.size, 3), dtype=np.uint8)
            img = Image.fromarray(arr)
        img = img.resize((self.size, self.size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0
        gray = (arr.mean(axis=2) * 255).astype(np.uint8)
        att = compute_attention_map(gray)
        cover_att = np.concatenate([arr.transpose(2, 0, 1), att[None, ...]], 0)
        return torch.from_numpy(cover_att).float(), torch.from_numpy(arr.transpose(2, 0, 1)).float()

    def __getitem__(self, idx):
        try:
            return self._load(idx)
        except Exception:
            arr = np.random.randint(0, 256, (self.size, self.size, 3), dtype=np.uint8).astype(np.float32)/255.0
            att = np.zeros((self.size, self.size), np.float32)
            cover_att = np.concatenate([arr.transpose(2,0,1), att[None,...]], 0)
            return torch.from_numpy(cover_att).float(), torch.from_numpy(arr.transpose(2,0,1)).float()

train_ds = HybridImageDataset(train_paths, size=IMG_SIZE)
val_ds = HybridImageDataset(val_paths if val_paths else train_paths[:max(1,len(train_paths)//10)], size=IMG_SIZE)
print('Train', len(train_ds), 'Val', len(val_ds), 'size', IMG_SIZE)


In [ ]:
# ===== CELL 09: Build model + RESUME from best checkpoint =====
def ssim_loss(x, y):
    C1, C2 = 0.01**2, 0.03**2
    mu_x = F.avg_pool2d(x, 3, 1, 1)
    mu_y = F.avg_pool2d(y, 3, 1, 1)
    sigma_x = F.avg_pool2d(x**2, 3, 1, 1) - mu_x**2
    sigma_y = F.avg_pool2d(y**2, 3, 1, 1) - mu_y**2
    sigma_xy = F.avg_pool2d(x*y, 3, 1, 1) - mu_x*mu_y
    ssim_map = ((2*mu_x*mu_y+C1)*(2*sigma_xy+C2))/((mu_x**2+mu_y**2+C1)*(sigma_x+sigma_y+C2)+1e-8)
    return 1 - ssim_map.mean()

BASE = 32
INN_CH = 8
N_BLOCKS = 4
model = ARESHybridINN(base=BASE, inn_channels=INN_CH, n_inn_blocks=N_BLOCKS).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

start_epoch = 1
best_val = float('inf')
history = []

CKPT_CANDIDATES = [
    PATHS['checkpoints'] / 'ares_hybrid_inn_best.pt',
    PATHS['checkpoints'] / 'ares_hybrid_inn_latest.pt',
    PATHS['exports'] / 'ares_hybrid_inn_best.pt',
    CODE / 'models/hybrid/ares_hybrid_inn_best.pt' if CODE.exists() else Path('/nonexistent'),
    CODE / 'models/hybrid/ares_hybrid_inn.pt' if CODE.exists() else Path('/nonexistent'),
]
ckpt_path = next((p for p in CKPT_CANDIDATES if p.exists()), None)

if ckpt_path is not None:
    print('RESUMING from', ckpt_path)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    state = ckpt.get('model', ckpt.get('hybrid', ckpt))
    missing, unexpected = model.load_state_dict(state, strict=False)
    print('load strict=False | missing', len(missing), 'unexpected', len(unexpected))
    if 'optimizer' in ckpt:
        try:
            opt.load_state_dict(ckpt['optimizer'])
            print('Optimizer state restored')
        except Exception as e:
            print('Optimizer not restored:', e)
    start_epoch = int(ckpt.get('epoch', 0)) + 1
    best_val = float(ckpt.get('best_val', ckpt.get('stats', {}).get('loss', best_val)))
    history = ckpt.get('history', [])
    print(f'Resume at epoch {start_epoch}, best_val={best_val}')
else:
    print('No checkpoint found — training FROM SCRATCH')
    print('After first save, re-runs will resume automatically.')

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')


In [ ]:
# ===== CELL 10: Training loop (saves every epoch to Drive) =====
EPOCHS = 30          # increase for full training (e.g. 50–100)
BATCH = 16 if DEVICE.type == 'cuda' else 4
ACCUM = 2            # gradient accumulation
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)

LAM = dict(mse=10.0, l1=5.0, ssim=2.0, residual=1.0)

def run_epoch(loader, train=True):
    model.train(train)
    totals = {k: 0.0 for k in ['loss','mse','l1','ssim','res','psnr']}
    n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    opt.zero_grad(set_to_none=True)
    with ctx:
        for step, (cover_att, cover) in enumerate(loader):
            cover_att = cover_att.to(DEVICE, non_blocking=True)
            cover = cover.to(DEVICE, non_blocking=True)
            B = cover.shape[0]
            rate = torch.rand(B, 1, device=DEVICE) * 0.45 + 0.05
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                if HAS_PROJECT:
                    residual, mask, *_ = model(cover_att, rate)
                else:
                    residual, mask = model(cover_att, rate)
                # residual only on blue channel, bounded
                delta = (0.4 * mask * residual).clamp(-0.04, 0.04)
                stego = cover.clone()
                stego[:, 2:3] = (cover[:, 2:3] + delta).clamp(0, 1)
                L_mse = F.mse_loss(stego, cover)
                L_l1 = F.l1_loss(stego, cover)
                L_ssim = ssim_loss(stego, cover)
                L_res = residual.abs().mean() + (residual**2).mean()
                loss = (LAM['mse']*L_mse + LAM['l1']*L_l1 + LAM['ssim']*L_ssim + LAM['residual']*L_res) / ACCUM
            if train:
                scaler.scale(loss).backward()
                if (step + 1) % ACCUM == 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)
            mse_v = float(L_mse.item())
            psnr = 99.0 if mse_v < 1e-12 else 10 * math.log10(1.0 / mse_v)
            totals['loss'] += float(loss.item() * ACCUM)
            totals['mse'] += mse_v
            totals['l1'] += float(L_l1.item())
            totals['ssim'] += float(L_ssim.item())
            totals['res'] += float(L_res.item())
            totals['psnr'] += psnr
            n += 1
            if step % 50 == 0:
                print(f"  step {step}/{len(loader)} loss={totals['loss']/max(1,n):.4f} psnr≈{totals['psnr']/max(1,n):.2f}")
    return {k: v/max(1,n) for k,v in totals.items()}

print(f'Training epochs {start_epoch}..{start_epoch+EPOCHS-1} on {DEVICE}, batch={BATCH}')
for ep in range(start_epoch, start_epoch + EPOCHS):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    dt = time.time() - t0
    print(f"Epoch {ep}: train_loss={tr['loss']:.4f} train_psnr≈{tr['psnr']:.2f} | val_loss={va['loss']:.4f} val_psnr≈{va['psnr']:.2f} | {dt:.1f}s")
    history.append({'epoch': ep, 'train': tr, 'val': va, 'time': dt})

    ckpt = {
        'model': model.state_dict(),
        'optimizer': opt.state_dict(),
        'epoch': ep,
        'best_val': best_val,
        'history': history,
        'config': {'base': BASE, 'inn_channels': INN_CH, 'n_inn_blocks': N_BLOCKS, 'img_size': IMG_SIZE},
        'stats': va,
        'dataset_manifest': DATASET_MANIFEST,
        'split_info': split_info,
    }
    latest = PATHS['checkpoints'] / 'ares_hybrid_inn_latest.pt'
    torch.save(ckpt, latest)
    torch.save(ckpt, PATHS['checkpoints'] / f'epoch_{ep:03d}.pt')
    # keep only last 3 epoch snapshots to save Drive space
    old = sorted(PATHS['checkpoints'].glob('epoch_*.pt'))
    for o in old[:-3]:
        try: o.unlink()
        except: pass

    if va['loss'] < best_val:
        best_val = va['loss']
        ckpt['best_val'] = best_val
        best_path = PATHS['checkpoints'] / 'ares_hybrid_inn_best.pt'
        torch.save(ckpt, best_path)
        shutil.copy(best_path, PATHS['exports'] / 'ares_hybrid_inn_best.pt')
        shutil.copy(best_path, PATHS['exports'] / 'ares_hybrid_inn.pt')
        print(f'  ★ New BEST val_loss={best_val:.6f} → saved to Drive exports/')

    with open(PATHS['logs'] / 'train_history.json', 'w') as f:
        json.dump(history, f, indent=2)

print('Training finished.')
print('BEST checkpoint:', PATHS['checkpoints'] / 'ares_hybrid_inn_best.pt')
print('Export copy:    ', PATHS['exports'] / 'ares_hybrid_inn_best.pt')


In [ ]:
# ===== CELL 11: Quick measured PSNR check on val images =====
from models.hybrid.ares_hybrid_inn import hybrid_embed, hybrid_extract  # may fail if no project
import traceback

def quick_psnr_eval(n=8, size=256):
    # reload best
    best = PATHS['exports'] / 'ares_hybrid_inn_best.pt'
    if not best.exists():
        best = PATHS['checkpoints'] / 'ares_hybrid_inn_best.pt'
    m = ARESHybridINN(base=BASE, inn_channels=INN_CH, n_inn_blocks=N_BLOCKS).to(DEVICE)
    if best.exists():
        ck = torch.load(best, map_location=DEVICE, weights_only=False)
        m.load_state_dict(ck.get('model', ck), strict=False)
        m.eval()
        print('Loaded', best)
    paths = val_paths[:n] if val_paths else train_paths[:n]
    secret = 'ARES_RESEARCH_BENCHMARK_SECRET'
    rows = []
    for p in paths:
        try:
            cover = Image.open(p).convert('RGB').resize((size, size), Image.BILINEAR)
        except Exception:
            continue
        try:
            if HAS_PROJECT:
                stego, info = hybrid_embed(cover, secret, password='benchmark', hybrid_model=m, use_residual=True)
                rec = hybrid_extract(stego, password='benchmark')
                ok = rec == secret
                rows.append({'psnr': info['psnr'], 'mse': info['mse'], 'tp': ok, 'path': str(p)})
                print(f"  PSNR={info['psnr']:.3f} MSE={info['mse']:.6e} TP={ok}")
            else:
                print('Skip embed eval (project hybrid_embed not available)')
                break
        except Exception as e:
            print('eval error', e)
    if rows:
        ps = [r['psnr'] for r in rows]
        print(f"Mean PSNR={np.mean(ps):.3f} median={np.median(ps):.3f} n={len(ps)} TP={sum(r['tp'] for r in rows)}/{len(rows)}")
    return rows

try:
    quick_psnr_eval(8, 256)
except Exception:
    traceback.print_exc()


In [ ]:
# ===== CELL 12: Download best model to browser (optional) =====
from google.colab import files
best = PATHS['exports'] / 'ares_hybrid_inn_best.pt'
if best.exists():
    print('Best model on Drive:', best)
    print('Size MB:', round(best.stat().st_size / 1e6, 2))
    # Uncomment to download to your laptop:
    # files.download(str(best))
else:
    print('No best checkpoint yet — run training cell first.')

print('''
=== HOW TO RESUME LATER ===
1. Open this same notebook
2. Run cells 01–04, 07–09 (skip long downloads if data already on Drive)
3. Cell 09 auto-loads MyDrive/ARES_Hybrid_INN/checkpoints/ares_hybrid_inn_best.pt
4. Cell 10 continues from last epoch

=== COPY BEST MODEL INTO PROJECT ===
On local machine after download:
  cp ares_hybrid_inn_best.pt ARES-Upgraded/models/hybrid/
  cp ares_hybrid_inn_best.pt ARES-Upgraded/models/hybrid/ares_hybrid_inn.pt
''')
